# OpenPlaque — BACCE/DDQN compatibility test

This notebook starts from `main` and is self-contained. It does not use `%run`.

Goal: verify that the public BACCE architecture can ingest OpenPlaque source CCTA series 7 and our automatically localized RCA-ostium neighborhood in exactly the patch/seed format expected by the DDQN tracker and branch detector.

It does **not** claim to run BACCE inference unless the original pretrained checkpoint files are supplied.


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque /content/BACCE
!git clone -q --branch bacce-compatibility-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git clone -q https://github.com/514sz/Branch-aware-centerline-extraction.git /content/BACCE

%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas torch

print('Repositories and packages ready.')


In [ ]:
import os, sys, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage as ndi
import torch

SRC = Path('/content/OpenPlaque/src')
BACCE = Path('/content/BACCE')
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(BACCE))

from openplaque.study import OpenPlaqueStudy
from Net import Tracker_Net, Detector_Net

ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTDIR = ROOT / 'BACCE_Compatibility'
OUTDIR.mkdir(parents=True, exist_ok=True)

print('torch:', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 1. Load source CCTA series 7 and cached TotalSegmentator aorta mask

This uses the successful aorta mask already produced by the TotalSegmentator validation. If the cache is missing, the notebook stops rather than silently reverting to a hand-built aorta detector.


In [ ]:
DRIVE_ZIP = ROOT/'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_bacce_compat'

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print('Copying Full_DICOM.zip locally...')
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
source_img, source, _ = study.load_series(7)
source = np.asarray(source)
spacing_xyz = np.array(source_img.GetSpacing(), float)
spacing_zyx = spacing_xyz[::-1]
origin = np.array(source_img.GetOrigin(), float)
direction = np.array(source_img.GetDirection(), float).reshape(3,3)

AORTA_CANDIDATES = [
    ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
    ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz',
]
aorta_path = next((p for p in AORTA_CANDIDATES if p.exists()), None)
if aorta_path is None:
    raise FileNotFoundError('Cached TotalSegmentator aorta mask not found. Re-run the successful TotalSegmentator validation first.')

aorta_img = sitk.ReadImage(str(aorta_path))
if aorta_img.GetSize() != source_img.GetSize() or not np.allclose(aorta_img.GetSpacing(), source_img.GetSpacing()):
    aorta_img = sitk.Resample(aorta_img, source_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
aorta = sitk.GetArrayFromImage(aorta_img) > 0

print('series 7 shape z,y,x:', source.shape)
print('spacing x,y,z mm:', tuple(spacing_xyz))
print('aorta mask:', aorta_path)
print('aorta voxels:', int(aorta.sum()))


## 2. Automatically recover the RCA-like ostium neighborhood

For this compatibility test we search only the already-validated root band (roughly z 225–305 in this scan), then rank small high-contrast components near the aortic surface on the patient-right side in LPS coordinates.


In [ ]:
def index_xyz_to_lps(x,y,z):
    return origin + direction @ (np.array([x,y,z],float) * spacing_xyz)

zlo, zhi = 225, 305
zlo=max(0,zlo); zhi=min(source.shape[0]-1,zhi)

dist = ndi.distance_transform_edt(~aorta, sampling=spacing_zyx)
shell = (dist > 0.8) & (dist <= 8.0)
band = np.zeros_like(aorta, bool)
band[zlo:zhi+1] = True

a_hu = source[aorta & band]
blood_thr = float(np.clip(np.median(a_hu)*0.43, 180, 360))
cand_mask = (source >= blood_thr) & shell & band
cand_mask = ndi.binary_closing(cand_mask, structure=np.ones((3,3,3),bool), iterations=1)
lab, n = ndi.label(cand_mask, structure=np.ones((3,3,3),bool))

aorta_centers={}
for z in range(zlo,zhi+1):
    yy,xx=np.where(aorta[z])
    if len(xx):
        aorta_centers[z]=index_xyz_to_lps(float(np.mean(xx)), float(np.mean(yy)), z)

rows=[]
for k in range(1,n+1):
    c=np.argwhere(lab==k)
    if len(c)<5:
        continue
    zmin,ymin,xmin=c.min(0); zmax,ymax,xmax=c.max(0)+1
    vol=len(c)*float(np.prod(spacing_xyz))
    if vol>2500:
        continue
    dv=dist[tuple(c.T)]
    if dv.min()>3.0 or dv.max()<2.0:
        continue
    mm=c.astype(float)*spacing_zyx
    if len(mm)>=3:
        ev, vec=np.linalg.eigh(np.cov(mm,rowvar=False))
        order=np.argsort(ev)[::-1]
        ev=ev[order]; vec=vec[:,order]
    else:
        continue
    length=float(np.sqrt(max(1e-6,12*ev[0])))
    elong=float(ev[0]/max(ev[1],1e-6))
    sub=(lab[zmin:zmax,ymin:ymax,xmin:xmax]==k)
    rmax=float(ndi.distance_transform_edt(sub,sampling=spacing_zyx).max())
    contact=c[np.argmin(dv)]
    zc=int(contact[0])
    if zc not in aorta_centers:
        continue
    p=index_xyz_to_lps(float(contact[2]),float(contact[1]),float(contact[0]))
    dx=float(p[0]-aorta_centers[zc][0])
    mean_hu=float(source[tuple(c.T)].mean())
    score = (
        0.28*np.tanh(length/10) +
        0.18*np.tanh(elong/5) +
        0.18*np.exp(-((rmax-2.0)/2.2)**2) +
        0.18*np.exp(-float(dv.min())/1.5) +
        0.18*np.tanh(max(0,-dx)/8)
    )
    rows.append(dict(label=k,score=score,length_mm=length,elong=elong,rmax_mm=rmax,
                     min_d_mm=float(dv.min()),max_d_mm=float(dv.max()),mean_hu=mean_hu,
                     dx_mm=dx,z=zc,y=float(contact[1]),x=float(contact[2])))

cand=pd.DataFrame(rows)
if cand.empty:
    raise RuntimeError('No RCA-like near-aortic components found.')
cand=cand.sort_values('score',ascending=False).reset_index(drop=True)
display(cand.head(10))

r1=cand.iloc[0]
seed=np.array([r1.z,r1.y,r1.x],float)
print('Selected compatibility seed z,y,x:', seed)


## 3. Estimate an initial BACCE direction and extract the exact 19×19×19 patch

BACCE's published code uses a 19³ patch (`gap=9`) and derives a start direction from two nearby centerline points. Here we estimate that direction from the principal axis of the local candidate component and orient it away from the aorta.


In [ ]:
k=int(r1.label)
coords=np.argwhere(lab==k)
mm=coords.astype(float)*spacing_zyx
center=mm.mean(0)
_,vec=np.linalg.eigh(np.cov(mm,rowvar=False))
axis=vec[:,-1]
z=int(round(seed[0]))
yy,xx=np.where(aorta[z])
a_ctr=np.array([z,float(np.mean(yy)),float(np.mean(xx))])*spacing_zyx
seed_mm=seed*spacing_zyx
if np.dot(axis, seed_mm-a_ctr) < 0:
    axis=-axis

step_mm=2.0
seed2_mm=seed_mm + axis*step_mm
seed2=seed2_mm/spacing_zyx
start_direction=seed2-seed

def clip_patch(arr, s, gap=9):
    s=np.round(s).astype(int)
    z,y,x=s
    if z-gap<0 or y-gap<0 or x-gap<0 or z+gap>=arr.shape[0] or y+gap>=arr.shape[1] or x+gap>=arr.shape[2]:
        raise ValueError('Seed too close to volume edge for BACCE 19^3 patch')
    return arr[z-gap:z+gap+1, y-gap:y+gap+1, x-gap:x+gap+1]

patch=clip_patch(source, seed, gap=9).astype(np.float32)
assert patch.shape==(19,19,19)
patch_t=torch.from_numpy(patch)[None,None]

print('seed1 z,y,x:', np.round(seed,2))
print('seed2 z,y,x:', np.round(seed2,2))
print('start direction:', np.round(start_direction,3))
print('patch shape:', tuple(patch_t.shape))
print('patch HU range:', float(patch.min()), 'to', float(patch.max()))


## 4. BACCE architecture smoke test

This instantiates the public `Tracker_Net` and `Detector_Net` without weights and verifies that the OpenPlaque patch produces the expected output shapes.


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tracker=Tracker_Net(n_actions=500).to(device).eval()
detector=Detector_Net().to(device).eval()

with torch.no_grad():
    q=tracker(patch_t.to(device))
    d=detector(patch_t.to(device))

print('Tracker output shape:', tuple(q.shape))
print('Detector output shape:', tuple(d.shape))

assert tuple(q.shape)==(1,501,1,1,1), 'Unexpected tracker output shape'
assert tuple(d.shape)==(1,3,1,1,1), 'Unexpected detector output shape'
print('BACCE architecture compatibility: PASS')


## 5. Optional checkpoint smoke test

Place the original files, if recovered, in:

- `MyDrive/OpenPlaque/BACCE_Checkpoints/Tracker.pth`
- `MyDrive/OpenPlaque/BACCE_Checkpoints/Detector.pth`

The notebook will load them only if both exist. No checkpoint is bundled or assumed.


In [ ]:
CKPT=ROOT/'BACCE_Checkpoints'
tracker_ckpt=CKPT/'Tracker.pth'
detector_ckpt=CKPT/'Detector.pth'

if tracker_ckpt.exists() and detector_ckpt.exists():
    tracker.load_state_dict(torch.load(tracker_ckpt,map_location=device))
    detector.load_state_dict(torch.load(detector_ckpt,map_location=device))
    tracker.eval(); detector.eval()
    with torch.no_grad():
        q=tracker(patch_t.to(device)).flatten()
        d=detector(patch_t.to(device)).flatten()
        prob=torch.softmax(q,dim=0)
        topv,topi=torch.topk(prob,5)
    print('CHECKPOINTS LOADED.')
    print('Top tracker actions:', list(zip(topi.cpu().tolist(), topv.cpu().tolist())))
    print('Detector raw outputs [bifurcation,end,radius]:', d.cpu().tolist())
else:
    print('No BACCE checkpoints found; architecture/input compatibility test completed without weights.')


## 6. Save a visual compatibility report

The plot shows the automatically recovered seed neighborhood and the 19³ BACCE patch center.


In [ ]:
z=int(round(seed[0])); y=int(round(seed[1])); x=int(round(seed[2]))
fig,axs=plt.subplots(1,3,figsize=(15,5))
axs[0].imshow(source[z],cmap='gray',vmin=-200,vmax=900)
axs[0].contour(aorta[z].astype(float),levels=[.5],linewidths=1.3)
axs[0].plot(x,y,'o')
axs[0].set_title(f'BACCE seed full context z={z}')
axs[0].axis('off')

r=65; y0=max(0,y-r);y1=min(source.shape[1],y+r);x0=max(0,x-r);x1=min(source.shape[2],x+r)
axs[1].imshow(source[z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=900)
axs[1].plot(x-x0,y-y0,'o')
axs[1].set_title('Seed close-up')
axs[1].axis('off')

axs[2].imshow(patch[9],cmap='gray',vmin=-200,vmax=900)
axs[2].plot(9,9,'o')
axs[2].set_title('BACCE 19×19×19 patch, center slice')
axs[2].axis('off')

plt.tight_layout()
out=OUTDIR/'bacce_compatibility_report.png'
fig.savefig(out,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

summary=pd.DataFrame([{
    'seed_z':seed[0],'seed_y':seed[1],'seed_x':seed[2],
    'seed2_z':seed2[0],'seed2_y':seed2[1],'seed2_x':seed2[2],
    'candidate_score':float(r1.score),
    'candidate_length_mm':float(r1.length_mm),
    'candidate_radius_mm':float(r1.rmax_mm),
    'tracker_output_shape':str(tuple(q.shape)),
    'detector_output_shape':str(tuple(d.shape)),
    'checkpoints_present':bool(tracker_ckpt.exists() and detector_ckpt.exists())
}])
display(summary)
summary.to_csv(OUTDIR/'bacce_compatibility_summary.csv',index=False)
print('Saved:',out)
print('BACCE compatibility notebook complete.')
